In [1]:
%reload_ext autoreload
%autoreload 2
# notebooks/result_processing.ipynb

# ---
# # Experiment Results Analysis
# This notebook orchestrates the loading, analysis, validation, and reporting 
# of experiment results logged to Weights & Biases.
# ---

# ## 1. Setup and Imports
# Import necessary libraries and custom modules. The `sys.path.append` line
# allows the notebook to find the custom `lmetk_report` package.
import sys
import pandas as pd
sys.path.append('..')

from lmetk_report.run_processor import load_runs_df
from lmetk_report.analysis import analyze_results
from lmetk_report.reporting import (
    create_summary_table,
    display_experiment_stats,
    display_performance_tables,
    display_hp_impact
)

# ---
# ## 2. Configuration
# Define all key parameters for the analysis in one place.
# ---
PROJECT_NAME = "01_2026_hsf15_valve_diagnostics_"
USER = "imos-buildingenergy-1"

# Metrics to extract
METRIC_PREFIXES = ["val/", "test/", "test_best_rerun/", "val_best_rerun/"]
REPORTING_METRICS = ["loss", "mse", "mae", "rmse", "f1", "accuracy"]

# --- NEW: Scientific Rigor Settings ---
# If defined, the analysis will DROP any configuration group that does not 
# contain exactly these seeds. Set to `None` to include all available data.

# Data seeds are set to None (Flexible mode)
REQUIRED_DATA_SEEDS = None      

# Model seeds are strictly enforced (Strict mode)
REQUIRED_MODEL_SEEDS = {72, 88, 101, 666, 226688}

# Columns to ignore if they vary within a group (avoids "Hidden Variation" warnings)
ADDITIONAL_IGNORED_COLS = ["gpu_id", "slurm_job_id"] 

# Configure pandas display options for better readability in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---
# ## 3. Data Loading
# Fetch data from wandb (or a local cache) and prepare the initial DataFrame.
# The `load_runs_df` function handles all the complex cleaning and flattening.
# ---
print("🚀 Loading Run Data...")
df, config_columns, dropped_columns = load_runs_df(
    project_name=PROJECT_NAME, 
    user=USER
)
print(f"Loaded {len(df)} runs.")

# ---
# ## 4. Core Analysis
# Run the main analysis function. This now includes:
# 1. Grouping by Model + Hyperparameters
# 2. Strict Seed Validation (Filtering incomplete groups)
# 3. Hidden Variation Detection (Warning about silent bugs)
# 4. Aggregation & Best Model Selection
# ---
print("\n🔬 Analyzing Results & Validating Consistency...")

all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    dropped_columns=dropped_columns,
    reporting_metrics=REPORTING_METRICS,
    metric_prefixes=METRIC_PREFIXES,
    # --- New Validation Arguments ---
    required_data_seeds=REQUIRED_DATA_SEEDS,
    required_model_seeds=REQUIRED_MODEL_SEEDS,
    additional_ignored_cols=ADDITIONAL_IGNORED_COLS
)

# ---
# ## 5. Reporting & Visualizations
# ---

# ### A. Experiment Statistics
# Shows how many configurations completed successfully and 
# confirms which seeds were actually found in the data.
display_experiment_stats(all_results)

# ### B. Performance Leaderboards
# Displays a separate, clean pivot table for every metric found (MSE, MAE, etc.).
display_performance_tables(all_results)

# ### C. Hyperparameter Impact Analysis
# Shows the top 10 configurations for each model, exposing exactly 
# which hyperparameters led to the best results.
display_hp_impact(all_results)

# ### D. Legacy Summary Table
# A single flattened table containing all metrics (useful for copying to Excel/LaTeX).
print("\n📊 Full Summary Table:")
summary_df = create_summary_table(all_results)
display(summary_df)

🚀 Loading Run Data...
Loaded 390 runs from cached file: csv_files/01_2026_hsf15_valve_diagnostics_.csv
Dropped 41 columns with >90% NaN values.
The columns were drropped: ['ckpt_path', 'pad_value', 'null_value', 'subset_seed', 'target_mask', 'profiler_cfg', 'subset_range', 'val_subset_ratio', 'optimizer_factory', 'scheduler_factory', 'test_subset_ratio', 'variable_emb_type', 'train_subset_ratio', 'optimization.scheduler.gamma', 'optimization.scheduler.step_size', 'model.device', 'model.n_jobs', 'model.fit_mode', 'model.random_state', 'model.yield_strategy', 'model.model_cache_path', 'model.yield_batch_size', 'model.ignore_pretraining_limits', 'dataset.task_type', 'datamodule.subset_seed', 'datamodule.val_subset_ratio', 'datamodule.test_subset_ratio', 'datamodule.train_subset_ratio', 'callbacks.early_stopping.stopping_threshold', 'callbacks.early_stopping.divergence_threshold', 'callbacks.early_stopping.check_on_train_epoch_end', 'callbacks.model_checkpoint.every_n_epochs', 'callbacks.m

,Dataset,Model,Configs Completed,Seeds Found
0,HSF15,lmetk.model.wrappers.fit_predict_tabdpt_wrapper.FitPredictTabDPTWrapper,6,Model: 42
1,HSF15,lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,24,"Model: 72, 88, 101, 666, 226688"
2,HSF15,lmetk.model.wrappers.fit_predict_tabpfn_wrapper.FitPredictTabPFNWrapper,6,Model: 42



  METRIC: test_best_rerunaccuracy


Dataset,HSF15
Model,
lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,0.43 ± 0.00
lmetk.model.wrappers.fit_predict_tabdpt_wrapper.FitPredictTabDPTWrapper,nan ± nan
lmetk.model.wrappers.fit_predict_tabpfn_wrapper.FitPredictTabPFNWrapper,nan ± nan



  METRIC: test_best_rerunf1


...

,optimization.lr,task_definition.seq_len,datamodule.train_batch_size,val/loss_mean,val/f1_mean,val/accuracy_mean,test/loss_mean,test/loss_std,test/loss_count,test/f1_mean,test/accuracy_mean,test_best_rerun/loss_mean,test_best_rerun/f1_mean,test_best_rerun/accuracy_mean,val_best_rerun/loss_mean,val_best_rerun/f1_mean,val_best_rerun/accuracy_mean
23,0.0010,100,512,4.318756,0.179926,0.255859,0.199749,0.000000,3,0.336042,0.429429,0.199749,0.336042,0.429429,0.196014,0.298936,0.367188
14,0.0005,100,16,0.126165,0.157512,0.219531,0.215559,0.075234,15,0.309319,0.482282,0.215559,0.309319,0.482282,0.175776,0.321090,0.501563
12,0.0005,50,16,0.181811,0.202619,0.252288,0.255410,0.127806,15,0.389981,0.545692,0.255410,0.389981,0.545692,0.237603,0.410941,0.549020
22,0.0010,100,16,0.099765,0.183511,0.246875,0.355410,0.094761,15,0.357620,0.439039,0.355410,0.357620,0.439039,0.272880,0.364217,0.456250
20,0.0010,50,16,0.181432,0.207754,0.212418,0.366248,0.095998,15,0.316588,0.451697,0.366248,0.316588,0.451697,0.323324,0.329233,0.450980
16,0.0010,1,16,0.280105,0.157466,0.222723,0.550678,0.604585,15,0.364821,0.544753,0.550678,0.364821,0.544753,0.538057,0.365685,0.548732
2,0.0001,10,16,0.285344,0.163769,0.282081,0.601347,0.286348,15,0.273345,0.392908,0.601347,0.273345,0.392908,0.494233,0.280184,0.398266
6,0.0001,100,16,0.191509,0.172046,0.289844,0.658131,0.695045,15,0.296042,0.514715,0.658131,0.296042,0.514715,0.587058,0.292300,0.519531
10,0.0005,10,16,0.604938,0.206666,0.257225,0.894615,0.173484,15,0.340816,0.442553,0.894615,0.340816,0.442553,0.648330,0.356479,0.468786
15,0.0005,100,512,1.128910,0.184894,0.277344,1.286629,2.108980,5,0.241398,0.445646,1.286629,0.241398,0.445646,1.236629,0.279801,0.508594



>>> lmetk.model.wrappers.fit_predict_tabpfn_wrapper.FitPredictTabPFNWrapper on HSF15
    (Sorted by test/loss)


,task_definition.seq_len,task_definition.stride_train,val/loss_mean,val/f1_mean,val/accuracy_mean,test/loss_mean,test/loss_std,test/loss_count,test/f1_mean,test/accuracy_mean,test_best_rerun/loss_mean,test_best_rerun/f1_mean,test_best_rerun/accuracy_mean,val_best_rerun/loss_mean,val_best_rerun/f1_mean,val_best_rerun/accuracy_mean
0,1,1,NaN,NaN,NaN,1.0,NaN,1,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,5,1,NaN,NaN,NaN,1.0,NaN,1,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,10,5,NaN,NaN,NaN,1.0,NaN,1,0.996454,0.997630,NaN,NaN,NaN,NaN,NaN,NaN
3,20,5,NaN,NaN,NaN,1.0,NaN,1,0.982254,0.987864,NaN,NaN,NaN,NaN,NaN,NaN
4,50,50,NaN,NaN,NaN,1.0,NaN,1,0.168278,0.476440,NaN,NaN,NaN,NaN,NaN,NaN
5,100,100,NaN,NaN,NaN,1.0,NaN,1,0.164646,0.490964,NaN,NaN,NaN,NaN,NaN,NaN



📊 Full Summary Table:


Dataset                                                              HSF15                                                                                                                     \
Metric                                             test_best_rerunaccuracy      test_best_rerunf1    test_best_rerunloss           testaccuracy                 testf1               testloss   
Model                                                                                                                                                                                           
lmetk.baselines.spacetimeformer_model.Spacetime...   0.4294 ± 0.0000 (n=3)  0.3360 ± 0.0000 (n=3)  0.1997 ± 0.0000 (n=3)  0.4294 ± 0.0000 (n=3)  0.3360 ± 0.0000 (n=3)  0.1997 ± 0.0000 (n=3)   
lmetk.model.wrappers.fit_predict_tabdpt_wrapper...         nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)  0.9884 ± 0.0000 (n=4)  0.9846 ± 0.0000 (n=4)  1.0000 ± 0.0000 (n=4)   
lmetk.model.wrappers.fit_predict_tabpfn_wrapper...         nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)     1.0000 ± nan (n=1)     1.0000 ± nan (n=1)     1.0000 ± nan (n=1)   

Dataset                                                                                                                                                                                       
Metric                                             val_best_rerunaccuracy       val_best_rerunf1     val_best_rerunloss            valaccuracy                  valf1                valloss  
Model                                                                                                                                                                                         
lmetk.baselines.spacetimeformer_model.Spacetime...  0.3672 ± 0.0000 (n=3)  0.2989 ± 0.0000 (n=3)  0.1960 ± 0.0000 (n=3)  0.2559 ± 0.0586 (n=4)  0.1799 ± 0.0720 (n=4)  4.3188 ± 8.1603 (n=4)  
lmetk.model.wrappers.fit_predict_tabdpt_wrapper...        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)  
lmetk.model.wrappers.fit_predict_tabpfn_wrapper...        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)        nan ± nan (n=0)

In [2]:
# 1. Access the raw data for the specific model and dataset
model_key = 'lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster'
dataset_key = 'HSF15'

# This DF contains all runs for this model before aggregation
df_subset = all_results[dataset_key][model_key]['non_aggregated_df']

# 2. Filter for ONE specific configuration that shows count=15
# (Based on your screenshot: lr=0.0001, seq_len=10, batch_size=16)
mask = (
    (df_subset['optimization.lr'] == 0.0001) & 
    (df_subset['task_definition.seq_len'] == 10) & 
    (df_subset['datamodule.train_batch_size'] == 16)
)
group_runs = df_subset[mask]

# 3. Display identifiers to spot the duplicates
# 'created_at' will show you if these runs are from different times/days
cols_to_inspect = ['run_name', 'id', 'created_at', 'seed', 'datasource.parameters.data_seed']
valid_cols = [c for c in cols_to_inspect if c in group_runs.columns]

print(f"Found {len(group_runs)} runs for this configuration (Expected 5).")
print("Listing all runs in this group:")
#display(group_runs[valid_cols].sort_values('created_at'))

Found 15 runs for this configuration (Expected 5).
Listing all runs in this group:


In [3]:
group_runs[valid_cols]

,run_name,seed
25,debug+hsf15+valve+diagnostics+stf_c0f10b13,72
26,debug+hsf15+valve+diagnostics+stf_59db61cb,88
27,debug+hsf15+valve+diagnostics+stf_0fe4497b,666
28,debug+hsf15+valve+diagnostics+stf_6102e251,101
29,debug+hsf15+valve+diagnostics+stf_15625fc5,226688
157,debug+hsf15+valve+diagnostics+stf_f1c2fe48,72
158,debug+hsf15+valve+diagnostics+stf_39782614,88
159,debug+hsf15+valve+diagnostics+stf_1722e80a,101
160,debug+hsf15+valve+diagnostics+stf_e4f83bb0,666
161,debug+hsf15+valve+diagnostics+stf_8178777e,226688


In [ ]:
# 1. Isolate the specific group of 15 runs (from your screenshot)
# Adjust these values if you want to look at a different group
target_model = 'lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster'
mask = (
    (df['model._target_'] == target_model) &
    (df['optimization.lr'] == 0.0001) & 
    (df['task_definition.seq_len'] == 10) & 
    (df['datamodule.train_batch_size'] == 16)
)
group_runs = df[mask].copy()

# 2. Find "Hidden Variables"
# We calculate the number of unique values for EVERY column in this small group.
# If a column has > 1 unique value, it means it varies between these "duplicate" runs.
drop_cols = [] #"hydra_cfg", "_wandb", "metadata", "transforms", "evaluator.val.metric_names" 
# find all dict or list columns:
for col in group_runs:
    if group_runs[col].apply(lambda x: isinstance(x, (dict, list))).any():
        drop_cols.append(col)

group_runs.drop(columns=drop_cols, errors='ignore', inplace=True)

nunique = group_runs.nunique()
varying_cols = nunique[nunique > 1].index.tolist()

# 3. Add identifiers for context (if they aren't already varying)
context_cols = ['run_name', 'seed']
for c in context_cols:
    if c not in varying_cols and c in group_runs.columns:
        varying_cols.insert(0, c)

# 4. Display sorted by Seed
print(f"Found {len(group_runs)} runs in this group.")
print("Showing ONLY columns that differ between these runs:")

# Sort by seed to put duplicates next to each other
display(
    group_runs[varying_cols]
    .sort_values(by=['seed',])
    .style.set_properties(**{'text-align': 'left'})
)

Found 15 runs in this group.
Showing ONLY columns that differ between these runs:


KeyError: 'created_at'

In [ ]:


group_runs 

,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,_runtime,_step,_timestamp,_wandb,epoch,lr-AdamW,test/accuracy,test/f1,test/loss,test/precision,test/recall,test_best_rerun/accuracy,test_best_rerun/f1,test_best_rerun/loss,test_best_rerun/precision,test_best_rerun/recall,train/loss_epoch,train/loss_step,trainer/global_step,val/accuracy,val/f1,val/loss,val/precision,val/recall,val_best_rerun/accuracy,val_best_rerun/f1,val_best_rerun/loss,val_best_rerun/precision,val_best_rerun/recall,d_x,d_ff,d_yc,d_yt,loss,norm,seed,tags,test,train,head_f,base_lr,d_model,n_heads,verbose,d_layers,d_values,e_layers,exp_name,mask_y_c,metadata,use_time,hydra_cfg,task_type,use_given,use_revin,use_space,activation,dropout_ff,evaluators,pin_memory,transforms,use_tt_dec,use_tt_enc,attn_factor,dropout_emb,dropout_qkv,max_seq_len,num_threads,num_workers,shuffle_val,use_val_dec,use_val_enc,embed_method,pos_emb_type,shuffle_test,time_emb_dim,linear_window,shuffle_train,time_emb_type,class_loss_imp,d_queries_keys,n_timefeatures,performer_relu,recon_loss_imp,use_final_norm,val_batch_size,flash_attention,local_self_attn,prefetch_factor,start_token_len,test_batch_size,attention_method,dropout_attn_out,experiment_group,global_self_attn,inverted_encoder,local_cross_attn,performer_kernel,train_batch_size,attn_time_windows,global_cross_attn,use_batch_sampler,model/params/total,timetable_emb_type,dropout_attn_matrix,enable_progress_bar,recon_mask_drop_seq,recon_mask_skip_all,use_seasonal_decomp,recon_mask_drop_full,linear_shared_weights,model/params/trainable,recon_mask_max_seq_len,scheduler_Loss_monitor,initial_downsample_convs,recon_mask_drop_standard,use_shifted_time_windows,performer_redraw_interval,intermediate_downsample_convs,run_name,optimization.lr,optimization.start_lr,optimization.optimizer.lr,optimization.optimizer._target_,optimization.optimizer.weight_decay,optimization.scheduler.factor,optimization.scheduler.init_lr,optimization.scheduler.peak_lr,optimization.scheduler._target_,optimization.scheduler.patience,optimization.scheduler.warmup_steps,datasource.fold,datasource._target_,datasource.cache_dir,datasource.data_name,datasource.task_mode,datasource.num_classes,datasource.auxiliary_tasks,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,task_definition.task_type,task_definition.num_classes,task_definition.stride_train,task_definition.target_metric,task_definition.requires_training,task_definition.target_metric_mode,task_definition.model.data_requirements.input_tensors,model.d_x,model.d_ff,model.d_yc,model.d_yt,model.base_lr,model.d_model,model.n_heads,model._target_,model.d_layers,model.e_layers,model.task_type,model.use_revin,model.dropout_ff,model.dropout_emb,model.dropout_qkv,model.max_seq_len,model.pos_emb_type,model.time_emb_dim,model.time_emb_type,model.n_timefeatures,model.local_self_attn,model.start_token_len,model.global_self_attn,model.local_cross_attn,model.global_cross_attn,model.timetable_emb_type,model.dropout_attn_matrix,model.use_seasonal_decomp,model.metadata.lightning_model,dataset._target_,dataset._recursive_,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg._target_,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,dataset.dataset_cfg.task_type,datamodule._target_,datamodule.num_workers,datamodule.shuffle_val,datamodule.shuffle_test,datamodule.shuffle_train,datamodule.val_batch_size,datamodule.prefetch_factor,datamodule.test_batch_size,datamodule.train_batch_size,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.model_summary.max_depth,callbacks.early_stopping.mode,callbacks.early_stopping.strict,callbacks.early_stopping.monitor,callbacks.early_stopping.verbose,callbacks.early_stopping._target_,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.early_stopping.

In [ ]:
hydra_cfg, hydra_cfg

In [2]:
for key in all_results.keys():
    print(f"Dataset key: {key}")
    for key_2 in all_results[key].keys():
        print(f"  Model key: {key_2}")
        for sub_key in all_results[key][key_2].keys():
            #if pd data frame then display the head only
            if isinstance(all_results[key][key_2][sub_key], pd.DataFrame):
                print(f"    {sub_key}:")
                display(all_results[key][key_2][sub_key].head(2))
            # if dict proceed with keys 
            elif isinstance(all_results[key][key_2][sub_key], dict):
                # same for inside dict and repeat
                for inner_key in all_results[key][key_2][sub_key].keys():
                    # print the 
                    if isinstance(all_results[key][key_2][sub_key][inner_key], pd.DataFrame):
                        print(f"   {sub_key}   {inner_key}:")
                        display(all_results[key][key_2][sub_key][inner_key].head(2))
                    else:
                        print(f" {sub_key}     {inner_key}: {all_results[key][key_2][sub_key][inner_key]}")
                
            else:
                print(f"    {sub_key}: {all_results[key][key_2][sub_key]}")
            
            


Dataset key: NB14
  Model key: lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster
 best_hyperparameters     task_definition.seq_len: 10
 best_hyperparameters     model.max_seq_len: 11.0
 best_hyperparameters     dataset.dataset_cfg.seq_len: 10.0
 best_hyperparameters     datamodule.train_batch_size: 16
 best_hyperparameters     run_names: ['debug+nb14+prognostics+combined+stf_61a9cc80', 'debug+nb14+prognostics+combined+stf_d0831fd2', 'debug+nb14+prognostics+combined+stf_dec7c232', 'debug+nb14+prognostics+combined+stf_b7d718c4', 'debug+nb14+prognostics+combined+stf_efaff768', 'debug+nb14+prognostics+combined+stf_9d450055', 'debug+nb14+prognostics+combined+stf_0de9df77', 'debug+nb14+prognostics+combined+stf_cbd4c0be', 'debug+nb14+prognostics+combined+stf_881aa4df', 'debug+nb14+prognostics+combined+stf_341fa4f1', 'debug+nb14+prognostics+combined+stf_d6dfad92', 'debug+nb14+prognostics+combined+stf_f0e15a8a', 'debug+nb14+prognostics+combined+stf_6b42d1d8', 'debug+nb14+prognost

Model task_definition.seq_len model.max_seq_len dataset.dataset_cfg.seq_len datamodule.train_batch_size  val/loss                 test/loss            \
                                                                                                                                                            mean       std count      mean       std   
2  lmetk.baselines.spacetimeformer_model.Spacetim...                      10              11.0                        10.0                          16  0.927458  1.569833    45  0.004994  0.002730   
7  lmetk.baselines.spacetimeformer_model.Spacetim...                     100             101.0                       100.0                         512  3.923929  4.241883    31  0.007404  0.003877   

        test_best_rerun/loss                 val_best_rerun/loss                                                           run_name  
  count                 mean       std count                mean       std count                                               list  
2    43             0.004893  0.002709    45            0.022714  0.036451    16  [debug+nb14+prognostics+combined+stf_61a9cc80,...  
7    30             0.007377  0.003814    31            0.012309  0.004241     7  [debug+nb14+prognostics+combined+stf_98e72e62,...

    non_aggregated_df:


,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,_runtime,_step,_timestamp,_wandb,epoch,lr-AdamW,plots/test/pred_vs_target_unit_15,plots/test/pred_vs_target_unit_16,plots/test/pred_vs_target_unit_18,plots/test/pred_vs_target_unit_19,plots/test/pred_vs_target_unit_23,plots/test/pred_vs_target_unit_24,plots/test/pred_vs_target_unit_27,plots/test/pred_vs_target_unit_28,plots/test/pred_vs_target_unit_5,plots/test/pred_vs_target_unit_6,plots/test/pred_vs_target_unit_7,plots/test/pred_vs_target_unit_8,plots/val/pred_vs_target_unit_15,plots/val/pred_vs_target_unit_18,plots/val/pred_vs_target_unit_23,plots/val/pred_vs_target_unit_27,plots/val/pred_vs_target_unit_5,plots/val/pred_vs_target_unit_7,test/loss,test/mae_16_denormalized,test/mae_16_normalized,test/mae_19_denormalized,test/mae_19_normalized,test/mae_24_denormalized,test/mae_24_normalized,test/mae_28_denormalized,test/mae_28_normalized,test/mae_6_denormalized,test/mae_6_normalized,test/mae_8_denormalized,test/mae_8_normalized,test/mae_denormalized_mean,test/mae_normalized_mean,test/mape_16_denormalized,test/mape_16_normalized,test/mape_19_denormalized,test/mape_19_normalized,test/mape_24_denormalized,test/mape_24_normalized,test/mape_28_denormalized,test/mape_28_normalized,test/mape_6_denormalized,test/mape_6_normalized,test/mape_8_denormalized,test/mape_8_normalized,test/mape_denormalized_mean,test/mape_normalized_mean,test/mse_16_denormalized,test/mse_16_normalized,test/mse_19_denormalized,test/mse_19_normalized,test/mse_24_denormalized,test/mse_24_normalized,test/mse_28_denormalized,test/mse_28_normalized,test/mse_6_denormalized,test/mse_6_normalized,test/mse_8_denormalized,test/mse_8_normalized,test/mse_denormalized_mean,test/mse_normalized_mean,test/phm_score_16_denormalized,test/phm_score_16_normalized,test/phm_score_19_denormalized,test/phm_score_19_normalized,test/phm_score_24_denormalized,test/phm_score_24_normalized,test/phm_score_28_denormalized,test/phm_score_28_normalized,test/phm_score_6_denormalized,test/phm_score_6_normalized,test/phm_score_8_denormalized,test/phm_score_8_normalized,test/phm_score_denormalized_mean,test/phm_score_normalized_mean,test/rmse_16_denormalized,test/rmse_16_normalized,test/rmse_19_denormalized,test/rmse_19_normalized,test/rmse_24_denormalized,test/rmse_24_normalized,test/rmse_28_denormalized,test/rmse_28_normalized,test/rmse_6_denormalized,test/rmse_6_normalized,test/rmse_8_denormalized,test/rmse_8_normalized,test/rmse_denormalized_mean,test/rmse_normalized_mean,test_best_rerun/loss,test_best_rerun/mae_16_denormalized,test_best_rerun/mae_16_normalized,test_best_rerun/mae_19_denormalized,test_best_rerun/mae_19_normalized,test_best_rerun/mae_24_denormalized,test_best_rerun/mae_24_normalized,test_best_rerun/mae_28_denormalized,test_best_rerun/mae_28_normalized,test_best_rerun/mae_6_denormalized,test_best_rerun/mae_6_normalized,test_best_rerun/mae_8_denormalized,test_best_rerun/mae_8_normalized,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_16_denormalized,test_best_rerun/mape_16_normalized,test_best_rerun/mape_19_denormalized,test_best_rerun/mape_19_normalized,test_best_rerun/mape_24_denormalized,test_best_rerun/mape_24_normalized,test_best_rerun/mape_28_denormalized,test_best_rerun/mape_28_normalized,test_best_rerun/mape_6_denormalized,test_best_rerun/mape_6_normalized,test_best_rerun/mape_8_denormalized,test_best_rerun/mape_8_normalized,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_16_denormalized,test_best_rerun/mse_16_normalized,test_best_rerun/mse_19_denormalized,test_best_rerun/mse_19_normalized,test_best_rerun/mse_24_denormalized,test_best_rerun/mse_24_normalized,test_best_rerun/mse_28_denormalized,test_best_rerun/mse_28_normalized,test_best_rerun/mse_6_denormalized,test_best_rerun/mse_6_normalize

  Model key: lmetk.model.wrappers.fit_predict_tabdpt_wrapper.FitPredictTabDPTWrapper
 best_hyperparameters     task_definition.seq_len: 1
 best_hyperparameters     task_definition.stride_train: 1
 best_hyperparameters     model.model_cache_path: /workspace/experiments/2025-09-18_TabPHM/debug/runs/debug+nb14+prognostics+combined+tabdpt_fit_predict/2026-01-22_12-57-04/model_cache_dir
 best_hyperparameters     run_names: ['debug+nb14+prognostics+combined+tabdpt_fit_predict_068f193e']
 best_performance     optimized_on: {'metric': 'val/loss', 'strategy': 'min'}
 best_performance     metrics: defaultdict(<class 'dict'>, {'loss': {'val': {'mean': np.float64(nan), 'std': np.float64(nan), 'count': np.int64(0)}, 'test': {'mean': np.float64(1.0), 'std': np.float64(nan), 'count': np.int64(1)}, 'test_best_rerun': {'mean': np.float64(nan), 'std': np.float64(nan), 'count': np.int64(0)}, 'val_best_rerun': {'mean': np.float64(nan), 'std': np.float64(nan), 'count': np.int64(0)}}})
    best_run_names: [

Model task_definition.seq_len task_definition.stride_train                             model.model_cache_path val/loss           test/loss            \
                                                                                                                                                                 mean std count      mean std count   
0  lmetk.model.wrappers.fit_predict_tabdpt_wrappe...                       1                            1  /workspace/experiments/2025-09-18_TabPHM/debug...      NaN NaN     0       1.0 NaN     1   
1  lmetk.model.wrappers.fit_predict_tabdpt_wrappe...                       1                            1  /workspace/experiments/2025-09-18_TabPHM/debug...      NaN NaN     0       1.0 NaN     1   

  test_best_rerun/loss           val_best_rerun/loss                                                     run_name  
                  mean std count                mean std count                                               list  
0                  NaN NaN     0                 NaN NaN     0  [debug+nb14+prognostics+combined+tabdpt_fit_pr...  
1                  NaN NaN     0                 NaN NaN     0  [debug+nb14+prognostics+combined+tabdpt_fit_pr...

    non_aggregated_df:


,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,_runtime,_step,_timestamp,_wandb,epoch,lr-AdamW,plots/test/pred_vs_target_unit_15,plots/test/pred_vs_target_unit_16,plots/test/pred_vs_target_unit_18,plots/test/pred_vs_target_unit_19,plots/test/pred_vs_target_unit_23,plots/test/pred_vs_target_unit_24,plots/test/pred_vs_target_unit_27,plots/test/pred_vs_target_unit_28,plots/test/pred_vs_target_unit_5,plots/test/pred_vs_target_unit_6,plots/test/pred_vs_target_unit_7,plots/test/pred_vs_target_unit_8,plots/val/pred_vs_target_unit_15,plots/val/pred_vs_target_unit_18,plots/val/pred_vs_target_unit_23,plots/val/pred_vs_target_unit_27,plots/val/pred_vs_target_unit_5,plots/val/pred_vs_target_unit_7,test/loss,test/mae_16_denormalized,test/mae_16_normalized,test/mae_19_denormalized,test/mae_19_normalized,test/mae_24_denormalized,test/mae_24_normalized,test/mae_28_denormalized,test/mae_28_normalized,test/mae_6_denormalized,test/mae_6_normalized,test/mae_8_denormalized,test/mae_8_normalized,test/mae_denormalized_mean,test/mae_normalized_mean,test/mape_16_denormalized,test/mape_16_normalized,test/mape_19_denormalized,test/mape_19_normalized,test/mape_24_denormalized,test/mape_24_normalized,test/mape_28_denormalized,test/mape_28_normalized,test/mape_6_denormalized,test/mape_6_normalized,test/mape_8_denormalized,test/mape_8_normalized,test/mape_denormalized_mean,test/mape_normalized_mean,test/mse_16_denormalized,test/mse_16_normalized,test/mse_19_denormalized,test/mse_19_normalized,test/mse_24_denormalized,test/mse_24_normalized,test/mse_28_denormalized,test/mse_28_normalized,test/mse_6_denormalized,test/mse_6_normalized,test/mse_8_denormalized,test/mse_8_normalized,test/mse_denormalized_mean,test/mse_normalized_mean,test/phm_score_16_denormalized,test/phm_score_16_normalized,test/phm_score_19_denormalized,test/phm_score_19_normalized,test/phm_score_24_denormalized,test/phm_score_24_normalized,test/phm_score_28_denormalized,test/phm_score_28_normalized,test/phm_score_6_denormalized,test/phm_score_6_normalized,test/phm_score_8_denormalized,test/phm_score_8_normalized,test/phm_score_denormalized_mean,test/phm_score_normalized_mean,test/rmse_16_denormalized,test/rmse_16_normalized,test/rmse_19_denormalized,test/rmse_19_normalized,test/rmse_24_denormalized,test/rmse_24_normalized,test/rmse_28_denormalized,test/rmse_28_normalized,test/rmse_6_denormalized,test/rmse_6_normalized,test/rmse_8_denormalized,test/rmse_8_normalized,test/rmse_denormalized_mean,test/rmse_normalized_mean,test_best_rerun/loss,test_best_rerun/mae_16_denormalized,test_best_rerun/mae_16_normalized,test_best_rerun/mae_19_denormalized,test_best_rerun/mae_19_normalized,test_best_rerun/mae_24_denormalized,test_best_rerun/mae_24_normalized,test_best_rerun/mae_28_denormalized,test_best_rerun/mae_28_normalized,test_best_rerun/mae_6_denormalized,test_best_rerun/mae_6_normalized,test_best_rerun/mae_8_denormalized,test_best_rerun/mae_8_normalized,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_16_denormalized,test_best_rerun/mape_16_normalized,test_best_rerun/mape_19_denormalized,test_best_rerun/mape_19_normalized,test_best_rerun/mape_24_denormalized,test_best_rerun/mape_24_normalized,test_best_rerun/mape_28_denormalized,test_best_rerun/mape_28_normalized,test_best_rerun/mape_6_denormalized,test_best_rerun/mape_6_normalized,test_best_rerun/mape_8_denormalized,test_best_rerun/mape_8_normalized,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_16_denormalized,test_best_rerun/mse_16_normalized,test_best_rerun/mse_19_denormalized,test_best_rerun/mse_19_normalized,test_best_rerun/mse_24_denormalized,test_best_rerun/mse_24_normalized,test_best_rerun/mse_28_denormalized,test_best_rerun/mse_28_normalized,test_best_rerun/mse_6_denormalized,test_best_rerun/mse_6_normalize

In [4]:
display(all_results[key][key_2]['sorted_aggregated_results'])

Model                    model.kernels                   model.strides                 model.dilations                     model.output_channels  \
                                                                                                                                                                                                  
0  lmetk.model.autoencoder.wrapper.AutoencoderWra...                  (5, 5, 5, 5, 5)                 (1, 1, 1, 1, 1)                 (1, 2, 4, 8, 1)                      (16, 32, 64, 64, 64)   
2  lmetk.model.autoencoder.wrapper.AutoencoderWra...  (10, 7, 7, 7, 7, 7, 7, 7, 7, 7)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (1, 1, 1, 1, 1, 1, 1, 1, 1, 1)  (16, 32, 64, 64, 64, 16, 32, 64, 64, 64)   
1  lmetk.model.autoencoder.wrapper.AutoencoderWra...                  (7, 7, 5, 5, 5)                 (2, 2, 1, 1, 1)                 (1, 1, 1, 1, 1)                      (16, 32, 64, 64, 64)   

   val/loss       val/mse       val/mae      val/rmse     test/loss      test/mse      test/mae     test/rmse                                               run_name  
       mean std      mean std      mean std      mean std      mean std      mean std      mean std      mean std                                               list  
0  0.000058 NaN  0.000058 NaN  0.004978 NaN  0.007645 NaN  0.000058 NaN  0.000058 NaN  0.004978 NaN  0.007645 NaN  [pronostia+reconstruction+residual_autoencoder...  
2  0.000063 NaN  0.000063 NaN  0.005087 NaN  0.007926 NaN  0.000063 NaN  0.000063 NaN  0.005087 NaN  0.007926 NaN  [pronostia+reconstruction+residual_autoencoder...  
1  0.000082 NaN  0.000082 NaN  0.005265 NaN  0.009032 NaN  0.000082 NaN  0.000082 NaN  0.005265 NaN  0.009032 NaN  [pronostia+reconstruction+residual_autoencoder...

In [ ]:
for i in all_results[key][key_2]['non_aggregated_df']['callbacks.model_checkpoint.dirpath']:
    print(str(i))
    

/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_03-30-38/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_03-30-47/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_11-49-26/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_12-25-24/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_20-06-41/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_21-16-34/checkpoints/checkpoints
/home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-03_04-29-43/checkpoints/checkpoints
/home/levtel/project

In [45]:
all_results[key][key_2]['non_aggregated_df']

,_runtime,_step,_timestamp,_wandb,epoch,lr-Adam,train/loss_step,trainer/global_step,loss,seed,tags,test,cache,train,logger,trainer,exp_name,evaluator,hydra_cfg,pin_memory,transforms,num_workers,shuffle_val,shuffle_test,shuffle_train,val_batch_size,prefetch_factor,test_batch_size,experiment_group,train_batch_size,use_batch_sampler,model/params/total,model/params/trainable,run_name,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/mae,train/mse,train/rmse,val/loss,val/mae,val/mse,val/rmse,num_threads,optimizer._target_,optimizer.optimizer_id,optimizer.scheduler.scheduler_id,optimizer.scheduler.scheduler_params.gamma,optimizer.scheduler.scheduler_params.step_size,optimizer.parameters.lr,optimizer.parameters.weight_decay,datasource._target_,datasource.data_dir,datasource.data_name,datasource.task_mode,datasource.fold,datasource.cache_dir,task_definition.stride,task_definition.seq_len,task_definition.pred_len,task_definition.label_len,task_definition.task_type,task_definition.requires_training,task_definition.model.data_requirements.input_tensors,model.kernels,model.strides,model._target_,model.dilations,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,model.output_channels,model.blocks_config,dataset._target_,dataset._recursive_,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg._target_,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,datamodule._target_,datamodule.num_workers,datamodule.shuffle_val,datamodule.shuffle_test,datamodule.shuffle_train,datamodule.val_batch_size,datamodule.prefetch_factor,datamodule.test_batch_size,datamodule.train_batch_size,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.model_summary.max_depth,callbacks.early_stopping.mode,callbacks.early_stopping.strict,callbacks.early_stopping.monitor,callbacks.early_stopping.verbose,callbacks.early_stopping._target_,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.early_stopping.check_finite,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint.verbose,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.save_last,callbacks.model_checkpoint.save_top_k,callbacks.model_checkpoint.save_weights_only,callbacks.model_checkpoint.auto_insert_metric_name,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,Model
52,29813,31573,1.759431e+09,{'runtime': 29813},20,0.00500,6.167217e-05,1573580,{'_target_': 'lmetk.loss.default.MSELoss'},42,[dev],True,"{'use_cache_after_loading': True, 'use_cache_a...",True,{'wandb': {'id': 'pronostia+reconstruction+res...,"{'devices': [1], '_target_': 'lightning.pytorc...",pronostia/reconstruction/residual_autoencoder,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",True,{'scaler_features': {'metadata': {'fit_on': 't...,4,False,False,True,1024,10,1024,debug,128,True,183208,183208,pronostia+reconstruction+residual_autoencoder_...,0.015844,0.003353,1487.310064,21.117129,0.006290,0.001688,67.621402,2.877372,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...",1.429764e-04,0.007379,2.859528e-04,0.011957,6.198527e-02,0.012541,1.239705e-01,2.489684e-01,1.429764e-04,0.007379,2.859528e-04,0.011957,NaN,lmetk.optimizer.default.DefaultOptimizer,Adam,StepLR,0.5,10,0.0100,0,lmetk.data.datasources.phmd_pronostia.Pronosti...,NaN,PRONOSTIA,rul,0.0,/home/levtel/projects/TabPHM/datasets/phmd_cache,1,64,0,0,reconstruction,True,[features]

In [36]:
# If run_name is the first element in the tuple
